# s04 — Streaming Responses

**What this teaches:** the difference between `agentkit.RunOnce` (buffered) and `agentkit.RunOnceStream` (token-by-token). ADK already streams text chunks internally — this example just lets you *see* them arrive in real time.

**Concept anchor:** streaming is a property of how the runner emits events, not of the agent itself. The same model and the same loop can be consumed as a final-only string or as an incremental token feed — your application picks the shape.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [docs/providers.md](../../docs/providers.md)).
- A provider/model that supports server-side streaming (almost all of them do).

## 1. Imports

Identical to [s01_loop](../s01_loop/s01_loop.ipynb): no tools needed for a pure streaming demo.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
)

## 2. Helper

Same panic-based `must` so the kernel stays alive on errors.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Construct the model and agent

A bare agent — no tools, no instruction. Streaming is orthogonal to everything else the agent might do, so we keep the wiring minimal.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s04_stream",
    Description: "Streaming-text demo.",
    Model:       llm,
})
must(err)

## 4. Build the runner

Same runner you've used in every prior example. What changes is *which* run-helper we invoke next.

In [ ]:
r, err := agentkit.Runner("s04", a)
must(err)

## 5. Run with streaming

Note the function name: `RunOnceStream` (not `RunOnce`). It emits partial text events as they arrive from the provider — you should literally watch the haiku materialise line by line in the cell output.

In [ ]:
prompt := "Stream a 50-line haiku about goroutines."
must(stream.Print(os.Stdout, agentkit.RunOnceStream(ctx, r, prompt)))

## What to look for

- Output appears in chunks, not all at once. Contrast with [s01_loop](../s01_loop/s01_loop.ipynb) where `RunOnce` waits for the full response.
- Streaming and tool calls compose: see [s05_tools](../s05_tools/s05_tools.ipynb) for an example that interleaves token streaming with tool dispatch.

## Try it yourself

1. Swap `RunOnceStream` back to `RunOnce` and re-run — output now appears as one block.
2. Set `OMNIS_DEBUG=1` before launching Jupyter to see per-chunk SSE timing in the event log.